# 🎯 Texture Analysis Pipeline v5
## Semi-Supervised Learning with Human Calibration

**Yenilikler:**
- Manuel labellama UI (300 görsel)
- XGBoost kalibrasyon modeli
- Label propagation (6000+ görsele)
- İnsan judgment + matematiksel features kombinasyonu

**Süreç:**
1. Dataset indir
2. Features çıkar
3. 300 görseli manuel labella (~30-45 dk)
4. Kalibrasyon modeli eğit
5. Tüm görsellere label ata
6. Ana model eğit

## 1. Setup & GPU Check

In [ ]:
# GPU kontrolü
!nvidia-smi

import tensorflow as tf
print(f"\nTensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Gerekli kütüphaneler
!pip install -q kaggle opencv-python mediapipe scikit-image PyWavelets xgboost ipywidgets

import os
import json
import random
import numpy as np
import pandas as pd
import cv2
from glob import glob
import shutil
from tqdm import tqdm

print("Kütüphaneler yüklendi!")

In [ ]:
# Kaggle API kurulumu
os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_credentials = {
    "username": "metekizilcik",
    "key": ""
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API hazır!")

## 2. Dataset Download

In [ ]:
%%writefile dataset_downloader_v5.py
"""Multi-dataset downloader for v5 pipeline"""

import os
import random
import shutil
from glob import glob

def download_datasets(base_dir="datasets", ffhq_count=5000, acne_count=1000, scar_count=500):
    """Download and organize datasets"""
    os.makedirs(base_dir, exist_ok=True)
    
    # FFHQ
    ffhq_dir = os.path.join(base_dir, "ffhq")
    if not os.path.exists(ffhq_dir) or len(glob(f"{ffhq_dir}/*.png")) < ffhq_count:
        print(f"FFHQ indiriliyor ({ffhq_count} görsel)...")
        os.system("kaggle datasets download -d greatgamedota/ffhq-face-data-set -p /tmp/ffhq --unzip -q")
        
        os.makedirs(ffhq_dir, exist_ok=True)
        all_ffhq = glob("/tmp/ffhq/**/*.png", recursive=True)
        selected = random.sample(all_ffhq, min(ffhq_count, len(all_ffhq)))
        
        for i, src in enumerate(selected):
            dst = os.path.join(ffhq_dir, f"ffhq_{i:05d}.png")
            shutil.copy2(src, dst)
        
        shutil.rmtree("/tmp/ffhq", ignore_errors=True)
        print(f"FFHQ: {len(glob(f'{ffhq_dir}/*.png'))} görsel")
    else:
        print(f"FFHQ zaten mevcut: {len(glob(f'{ffhq_dir}/*.png'))} görsel")
    
    # Acne Dataset
    acne_dir = os.path.join(base_dir, "acne")
    if not os.path.exists(acne_dir) or len(glob(f"{acne_dir}/*.*")) < 100:
        print("Acne dataset indiriliyor...")
        os.system("kaggle datasets download -d rutviklathiyateksun/acne-dataset-image-4620-images -p /tmp/acne --unzip -q")
        
        os.makedirs(acne_dir, exist_ok=True)
        all_acne = glob("/tmp/acne/**/*.jpg", recursive=True) + glob("/tmp/acne/**/*.jpeg", recursive=True) + glob("/tmp/acne/**/*.png", recursive=True)
        selected = random.sample(all_acne, min(acne_count, len(all_acne)))
        
        for i, src in enumerate(selected):
            ext = os.path.splitext(src)[1]
            dst = os.path.join(acne_dir, f"acne_{i:05d}{ext}")
            shutil.copy2(src, dst)
        
        shutil.rmtree("/tmp/acne", ignore_errors=True)
        print(f"Acne: {len(glob(f'{acne_dir}/*.*'))} görsel")
    else:
        print(f"Acne zaten mevcut: {len(glob(f'{acne_dir}/*.*'))} görsel")
    
    # Scar Dataset  
    scar_dir = os.path.join(base_dir, "scar")
    if not os.path.exists(scar_dir) or len(glob(f"{scar_dir}/*.*")) < 100:
        print("Scar dataset indiriliyor...")
        os.system("kaggle datasets download -d saranshbagri/scar -p /tmp/scar --unzip -q")
        
        os.makedirs(scar_dir, exist_ok=True)
        all_scar = glob("/tmp/scar/**/*.jpg", recursive=True) + glob("/tmp/scar/**/*.jpeg", recursive=True) + glob("/tmp/scar/**/*.png", recursive=True)
        selected = random.sample(all_scar, min(scar_count, len(all_scar)))
        
        for i, src in enumerate(selected):
            ext = os.path.splitext(src)[1]
            dst = os.path.join(scar_dir, f"scar_{i:05d}{ext}")
            shutil.copy2(src, dst)
        
        shutil.rmtree("/tmp/scar", ignore_errors=True)
        print(f"Scar: {len(glob(f'{scar_dir}/*.*'))} görsel")
    else:
        print(f"Scar zaten mevcut: {len(glob(f'{scar_dir}/*.*'))} görsel")
    
    # Özet
    total = 0
    for cat in ["ffhq", "acne", "scar"]:
        count = len(glob(f"{base_dir}/{cat}/*.*"))
        total += count
    
    print(f"\nToplam: {total} görsel")
    return base_dir

if __name__ == "__main__":
    download_datasets()

In [ ]:
# Datasetleri indir
from dataset_downloader_v5 import download_datasets

download_datasets(
    ffhq_count=5000,   # Normal yüzler
    acne_count=1000,   # Sivilceli
    scar_count=500     # Skar
)

## 3. Feature Extraction

In [ ]:
%%writefile feature_extractor_v5.py
"""Feature extraction for calibration model"""

import cv2
import numpy as np
from scipy.fftpack import fft2, fftshift
import pywt
from skimage.feature import graycomatrix, graycoprops

def create_texture_map(image_path):
    """Create texture map from image"""
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    img = cv2.resize(img, (256, 256))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel = cv2.split(lab)[0]
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(l_channel)
    
    blurred = cv2.GaussianBlur(enhanced, (37, 37), 0)
    texture_map = cv2.add(cv2.subtract(enhanced, blurred), 128)
    
    return texture_map

def compute_fft_entropy(texture_map):
    """2D FFT tabanlı entropy"""
    f_transform = fft2(texture_map.astype(np.float32))
    f_shift = fftshift(f_transform)
    magnitude = np.abs(f_shift)
    magnitude = magnitude / (np.sum(magnitude) + 1e-10)
    magnitude = magnitude[magnitude > 0]
    entropy = -np.sum(magnitude * np.log2(magnitude + 1e-10))
    return entropy

def compute_spatial_entropy(texture_map):
    """Spatial domain entropy"""
    hist, _ = np.histogram(texture_map, bins=256, range=(0, 256))
    hist = hist / (np.sum(hist) + 1e-10)
    hist = hist[hist > 0]
    entropy = -np.sum(hist * np.log2(hist + 1e-10))
    return entropy

def compute_dwt_energy(texture_map):
    """DWT energy (detail coefficients)"""
    coeffs = pywt.wavedec2(texture_map, 'sym4', level=3)
    detail_energy = 0
    for i in range(1, len(coeffs)):
        for detail in coeffs[i]:
            detail_energy += np.sum(detail ** 2)
    return np.log1p(detail_energy)

def compute_roughness(texture_map):
    """Ra ve Rq hesapla"""
    texture_float = texture_map.astype(np.float32)
    mean_val = np.mean(texture_float)
    deviations = texture_float - mean_val
    ra = np.mean(np.abs(deviations))
    rq = np.sqrt(np.mean(deviations ** 2))
    return ra, rq

def compute_mlc(texture_map, window_size=15):
    """Mean Local Contrast"""
    texture_float = texture_map.astype(np.float32)
    kernel = np.ones((window_size, window_size), np.float32) / (window_size ** 2)
    local_mean = cv2.filter2D(texture_float, -1, kernel)
    local_sq_mean = cv2.filter2D(texture_float ** 2, -1, kernel)
    local_std = np.sqrt(np.maximum(local_sq_mean - local_mean ** 2, 0))
    return np.mean(local_std)

def compute_glcm_features(texture_map):
    """GLCM features"""
    img_uint8 = (texture_map / 4).astype(np.uint8)  # 64 levels
    glcm = graycomatrix(img_uint8, distances=[1, 2], angles=[0, np.pi/4, np.pi/2], 
                        levels=64, symmetric=True, normed=True)
    
    contrast = np.mean(graycoprops(glcm, 'contrast'))
    homogeneity = np.mean(graycoprops(glcm, 'homogeneity'))
    energy = np.mean(graycoprops(glcm, 'energy'))
    correlation = np.mean(graycoprops(glcm, 'correlation'))
    
    return contrast, homogeneity, energy, correlation

def compute_laplacian_variance(texture_map):
    """Laplacian variance (blur detection)"""
    laplacian = cv2.Laplacian(texture_map, cv2.CV_64F)
    return laplacian.var()

def compute_gradient_magnitude(texture_map):
    """Average gradient magnitude"""
    sobelx = cv2.Sobel(texture_map, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(texture_map, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    return np.mean(magnitude)

def extract_features(image_path):
    """Extract all features from image"""
    texture_map = create_texture_map(image_path)
    if texture_map is None:
        return None
    
    # Compute all features
    fft_entropy = compute_fft_entropy(texture_map)
    spatial_entropy = compute_spatial_entropy(texture_map)
    dwt_energy = compute_dwt_energy(texture_map)
    ra, rq = compute_roughness(texture_map)
    mlc = compute_mlc(texture_map)
    contrast, homogeneity, energy, correlation = compute_glcm_features(texture_map)
    laplacian_var = compute_laplacian_variance(texture_map)
    gradient_mag = compute_gradient_magnitude(texture_map)
    
    # Feature vector
    features = {
        'fft_entropy': fft_entropy,
        'spatial_entropy': spatial_entropy,
        'dwt_energy': dwt_energy,
        'ra': ra,
        'rq': rq,
        'mlc': mlc,
        'glcm_contrast': contrast,
        'glcm_homogeneity': homogeneity,
        'glcm_energy': energy,
        'glcm_correlation': correlation,
        'laplacian_var': laplacian_var,
        'gradient_mag': gradient_mag
    }
    
    return features

def extract_features_batch(image_paths, show_progress=True):
    """Extract features for multiple images"""
    from tqdm import tqdm
    
    results = []
    iterator = tqdm(image_paths) if show_progress else image_paths
    
    for path in iterator:
        features = extract_features(path)
        if features is not None:
            features['image_path'] = path
            results.append(features)
    
    return results

In [ ]:
# Tüm görsellerin feature'larını çıkar
from feature_extractor_v5 import extract_features_batch
from glob import glob
import pandas as pd

# Tüm görselleri bul
all_images = []
for category in ['ffhq', 'acne', 'scar']:
    images = glob(f"datasets/{category}/*.*")
    for img in images:
        all_images.append({'path': img, 'category': category})

print(f"Toplam görsel: {len(all_images)}")

# Feature extraction
print("\nFeature extraction başlıyor...")
paths = [x['path'] for x in all_images]
features_list = extract_features_batch(paths)

# DataFrame oluştur
df = pd.DataFrame(features_list)

# Kategori bilgisi ekle
path_to_category = {x['path']: x['category'] for x in all_images}
df['category'] = df['image_path'].map(path_to_category)

# Kaydet
df.to_csv('all_features.csv', index=False)
print(f"\nFeatures çıkarıldı: {len(df)} görsel")
print(df.head())

## 4. Manuel Labellama UI

**Talimatlar:**
- Her görsel için 0-100 arası skor gir
- 0 = Çok kötü doku (ciddi sivilce, derin skar)
- 50 = Ortalama
- 100 = Mükemmel pürüzsüz cilt
- Her kategoriden 100 görsel labellanacak

In [ ]:
%%writefile manual_labeler.py
"""Manual labeling utilities for Colab"""

import os
import json
import random
from glob import glob
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import cv2
import numpy as np
from io import BytesIO
import base64

class ManualLabeler:
    def __init__(self, datasets_dir="datasets", samples_per_category=100, checkpoint_file="manual_labels.json"):
        self.datasets_dir = datasets_dir
        self.samples_per_category = samples_per_category
        self.checkpoint_file = checkpoint_file
        self.labels = {}
        self.current_idx = 0
        self.images_to_label = []
        
        # Checkpoint varsa yükle
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'r') as f:
                self.labels = json.load(f)
            print(f"Checkpoint yüklendi: {len(self.labels)} label mevcut")
        
        self._prepare_images()
    
    def _prepare_images(self):
        """Her kategoriden rastgele görsel seç"""
        categories = ['ffhq', 'acne', 'scar']
        
        for cat in categories:
            cat_images = glob(f"{self.datasets_dir}/{cat}/*.*")
            
            # Zaten labellanmamış olanları filtrele
            unlabeled = [img for img in cat_images if img not in self.labels]
            
            # Yeterli sayıda seç
            needed = self.samples_per_category - sum(1 for k in self.labels if f"/{cat}/" in k)
            if needed > 0:
                selected = random.sample(unlabeled, min(needed, len(unlabeled)))
                self.images_to_label.extend(selected)
        
        random.shuffle(self.images_to_label)
        print(f"Labellanacak görsel: {len(self.images_to_label)}")
    
    def _img_to_base64(self, img_path):
        """Görseli base64'e çevir"""
        img = cv2.imread(img_path)
        if img is None:
            return None
        img = cv2.resize(img, (400, 400))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        _, buffer = cv2.imencode('.jpg', img_rgb)
        return base64.b64encode(buffer).decode('utf-8')
    
    def save_checkpoint(self):
        """Labelları kaydet"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump(self.labels, f, indent=2)
    
    def get_progress(self):
        """İlerleme durumu"""
        total = self.samples_per_category * 3
        done = len(self.labels)
        return done, total
    
    def start_labeling(self):
        """Labellama UI'ı başlat"""
        if not self.images_to_label:
            print("Tüm görseller labellandı!")
            return
        
        # UI bileşenleri
        output = widgets.Output()
        image_widget = widgets.HTML()
        
        score_slider = widgets.IntSlider(
            value=50, min=0, max=100, step=5,
            description='Skor:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='400px')
        )
        
        score_input = widgets.BoundedIntText(
            value=50, min=0, max=100,
            description='veya:',
            style={'description_width': '40px'},
            layout=widgets.Layout(width='120px')
        )
        
        # Slider ve input senkronizasyonu
        widgets.jslink((score_slider, 'value'), (score_input, 'value'))
        
        submit_btn = widgets.Button(
            description='Kaydet & Sonraki',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        
        skip_btn = widgets.Button(
            description='Atla',
            button_style='warning',
            layout=widgets.Layout(width='80px')
        )
        
        progress_label = widgets.HTML()
        info_label = widgets.HTML()
        
        # Quick score buttons
        quick_buttons = []
        for score in [0, 20, 40, 60, 80, 100]:
            btn = widgets.Button(
                description=str(score),
                layout=widgets.Layout(width='50px')
            )
            btn.score = score
            quick_buttons.append(btn)
        
        def update_display():
            if self.current_idx >= len(self.images_to_label):
                image_widget.value = "<h2>Tamamlandı! 🎉</h2>"
                self.save_checkpoint()
                return
            
            img_path = self.images_to_label[self.current_idx]
            base64_img = self._img_to_base64(img_path)
            
            if base64_img:
                image_widget.value = f'<img src="data:image/jpeg;base64,{base64_img}" style="border: 2px solid #333;"/>'
            else:
                image_widget.value = "<p>Görsel yüklenemedi</p>"
            
            # Kategori ve ilerleme
            category = "ffhq" if "/ffhq/" in img_path else ("acne" if "/acne/" in img_path else "scar")
            done, total = self.get_progress()
            
            progress_label.value = f"<b>İlerleme:</b> {done}/{total} ({100*done/total:.1f}%)"
            info_label.value = f"<b>Kategori:</b> {category.upper()} | <b>Dosya:</b> {os.path.basename(img_path)}"
            
            score_slider.value = 50
        
        def on_submit(btn):
            if self.current_idx < len(self.images_to_label):
                img_path = self.images_to_label[self.current_idx]
                self.labels[img_path] = score_slider.value
                self.current_idx += 1
                
                # Her 10 labelde kaydet
                if len(self.labels) % 10 == 0:
                    self.save_checkpoint()
                
                update_display()
        
        def on_skip(btn):
            if self.current_idx < len(self.images_to_label):
                self.current_idx += 1
                update_display()
        
        def on_quick(btn):
            score_slider.value = btn.score
            on_submit(None)
        
        submit_btn.on_click(on_submit)
        skip_btn.on_click(on_skip)
        for btn in quick_buttons:
            btn.on_click(on_quick)
        
        # Layout
        quick_box = widgets.HBox(quick_buttons, layout=widgets.Layout(margin='10px 0'))
        score_box = widgets.HBox([score_slider, score_input])
        button_box = widgets.HBox([submit_btn, skip_btn])
        
        guide = widgets.HTML(value="""
        <div style="background: #f0f0f0; padding: 10px; margin: 10px 0; border-radius: 5px;">
        <b>Puanlama Rehberi:</b><br>
        • 0-20: Çok kötü (ciddi sivilce, derin skar, belirgin lezyon)<br>
        • 20-40: Kötü (görünür sivilce/skar, düzensiz doku)<br>
        • 40-60: Ortalama (hafif kusurlar, normal gözenek)<br>
        • 60-80: İyi (temiz cilt, minimal kusur)<br>
        • 80-100: Mükemmel (pürüzsüz, sağlıklı görünüm)
        </div>
        """)
        
        ui = widgets.VBox([
            progress_label,
            image_widget,
            info_label,
            guide,
            widgets.HTML("<b>Hızlı Skor:</b>"),
            quick_box,
            score_box,
            button_box
        ])
        
        update_display()
        display(ui)

In [ ]:
# Manuel labellama başlat
from manual_labeler import ManualLabeler

labeler = ManualLabeler(
    datasets_dir="datasets",
    samples_per_category=100,  # Her kategoriden 100 görsel
    checkpoint_file="manual_labels.json"
)

# UI'ı başlat
labeler.start_labeling()

In [ ]:
# Labellama durumunu kontrol et
import json
import os

if os.path.exists('manual_labels.json'):
    with open('manual_labels.json', 'r') as f:
        labels = json.load(f)
    
    print(f"Toplam label: {len(labels)}")
    
    # Kategori bazında
    for cat in ['ffhq', 'acne', 'scar']:
        count = sum(1 for k in labels if f"/{cat}/" in k)
        scores = [v for k, v in labels.items() if f"/{cat}/" in k]
        if scores:
            print(f"  {cat}: {count} label, ortalama skor: {np.mean(scores):.1f}")
else:
    print("Henüz label yok. Yukarıdaki hücreyi çalıştırın.")

## 5. Kalibrasyon Modeli (XGBoost)

Manuel labellardan öğrenerek tüm görsellere skor atayacak model

In [ ]:
%%writefile calibration_model.py
"""XGBoost calibration model for label propagation"""

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import joblib

class CalibrationModel:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.feature_columns = [
            'fft_entropy', 'spatial_entropy', 'dwt_energy',
            'ra', 'rq', 'mlc',
            'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation',
            'laplacian_var', 'gradient_mag'
        ]
    
    def prepare_training_data(self, features_df, labels_dict):
        """Manuel labellarla eşleşen feature'ları hazırla"""
        # Label'ı olan görselleri filtrele
        labeled_paths = set(labels_dict.keys())
        mask = features_df['image_path'].isin(labeled_paths)
        df_labeled = features_df[mask].copy()
        
        # Label'ları ekle
        df_labeled['manual_score'] = df_labeled['image_path'].map(labels_dict)
        
        print(f"Eğitim verisi: {len(df_labeled)} görsel")
        return df_labeled
    
    def train(self, df_labeled):
        """Kalibrasyon modelini eğit"""
        X = df_labeled[self.feature_columns].values
        y = df_labeled['manual_score'].values
        
        # Normalize
        X_scaled = self.scaler.fit_transform(X)
        
        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42
        )
        
        # XGBoost model
        self.model = XGBRegressor(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        )
        
        self.model.fit(X_train, y_train)
        
        # Evaluate
        train_pred = self.model.predict(X_train)
        test_pred = self.model.predict(X_test)
        
        train_mae = np.mean(np.abs(train_pred - y_train))
        test_mae = np.mean(np.abs(test_pred - y_test))
        
        print(f"Train MAE: {train_mae:.2f}")
        print(f"Test MAE: {test_mae:.2f}")
        
        # Cross-validation
        cv_scores = cross_val_score(self.model, X_scaled, y, cv=5, scoring='neg_mean_absolute_error')
        print(f"CV MAE: {-cv_scores.mean():.2f} (+/- {cv_scores.std()*2:.2f})")
        
        # Feature importance
        importance = self.model.feature_importances_
        print("\nFeature Importance:")
        for name, imp in sorted(zip(self.feature_columns, importance), key=lambda x: -x[1]):
            print(f"  {name}: {imp:.3f}")
        
        return test_mae
    
    def predict(self, features_df):
        """Tüm görsellere skor tahmin et"""
        X = features_df[self.feature_columns].values
        X_scaled = self.scaler.transform(X)
        
        predictions = self.model.predict(X_scaled)
        predictions = np.clip(predictions, 0, 100)
        
        return predictions
    
    def save(self, path="calibration_model.joblib"):
        """Model kaydet"""
        joblib.dump({
            'model': self.model,
            'scaler': self.scaler,
            'feature_columns': self.feature_columns
        }, path)
        print(f"Model kaydedildi: {path}")
    
    def load(self, path="calibration_model.joblib"):
        """Model yükle"""
        data = joblib.load(path)
        self.model = data['model']
        self.scaler = data['scaler']
        self.feature_columns = data['feature_columns']
        print(f"Model yüklendi: {path}")

In [ ]:
# Kalibrasyon modelini eğit
from calibration_model import CalibrationModel
import json

# Manuel labelları yükle
with open('manual_labels.json', 'r') as f:
    manual_labels = json.load(f)

print(f"Manuel label sayısı: {len(manual_labels)}")

# Feature'ları yükle
df_features = pd.read_csv('all_features.csv')

# Kalibrasyon modeli
calibrator = CalibrationModel()

# Eğitim verisi hazırla
df_labeled = calibrator.prepare_training_data(df_features, manual_labels)

# Eğit
calibrator.train(df_labeled)

# Kaydet
calibrator.save('calibration_model.joblib')

## 6. Label Propagation

Kalibrasyon modeliyle tüm görsellere skor atama

In [ ]:
# Tüm görsellere skor ata
from calibration_model import CalibrationModel

# Model yükle
calibrator = CalibrationModel()
calibrator.load('calibration_model.joblib')

# Tahmin yap
df_features = pd.read_csv('all_features.csv')
df_features['predicted_score'] = calibrator.predict(df_features)

# Manuel label olanları koru
with open('manual_labels.json', 'r') as f:
    manual_labels = json.load(f)

def get_final_score(row):
    if row['image_path'] in manual_labels:
        return manual_labels[row['image_path']]
    return row['predicted_score']

df_features['final_score'] = df_features.apply(get_final_score, axis=1)

# İstatistikler
print("\n=== SKOR DAĞILIMI ===")
print(f"Toplam görsel: {len(df_features)}")
print(f"Min: {df_features['final_score'].min():.1f}")
print(f"Max: {df_features['final_score'].max():.1f}")
print(f"Mean: {df_features['final_score'].mean():.1f}")
print(f"Std: {df_features['final_score'].std():.1f}")

print("\n=== KATEGORİ BAZINDA ===")
for cat in ['ffhq', 'acne', 'scar']:
    cat_scores = df_features[df_features['category'] == cat]['final_score']
    print(f"{cat.upper()}: mean={cat_scores.mean():.1f}, std={cat_scores.std():.1f}")

# Kaydet
df_features[['image_path', 'category', 'final_score']].to_csv('final_labels.csv', index=False)
print("\nLabellar kaydedildi: final_labels.csv")

In [ ]:
# Skor dağılımı görselleştir
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Genel dağılım
axes[0].hist(df_features['final_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Tüm Görseller')
axes[0].set_xlabel('Skor')
axes[0].set_ylabel('Sayı')

# Kategori bazında
colors = {'ffhq': 'green', 'acne': 'red', 'scar': 'orange'}
for i, cat in enumerate(['ffhq', 'acne', 'scar']):
    cat_scores = df_features[df_features['category'] == cat]['final_score']
    axes[i+1].hist(cat_scores, bins=20, edgecolor='black', alpha=0.7, color=colors[cat])
    axes[i+1].set_title(f'{cat.upper()} (n={len(cat_scores)})')
    axes[i+1].set_xlabel('Skor')

plt.tight_layout()
plt.savefig('score_distribution.png', dpi=100)
plt.show()

## 7. Texture Map Generation

In [ ]:
%%writefile texture_generator_v5.py
"""Generate texture maps for training"""

import cv2
import numpy as np
import os
from tqdm import tqdm

def create_texture_map(image_path, output_size=224):
    """Create texture map from image"""
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # Resize to 256 for processing
    img = cv2.resize(img, (256, 256))
    
    # LAB color space
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel = cv2.split(lab)[0]
    
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(l_channel)
    
    # Texture extraction
    blurred = cv2.GaussianBlur(enhanced, (37, 37), 0)
    texture_map = cv2.add(cv2.subtract(enhanced, blurred), 128)
    
    # Resize to output size
    texture_map = cv2.resize(texture_map, (output_size, output_size))
    
    return texture_map

def generate_texture_maps(df, output_dir="texture_maps"):
    """Generate texture maps for all images in dataframe"""
    os.makedirs(output_dir, exist_ok=True)
    
    results = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_path = row['image_path']
        score = row['final_score']
        
        texture_map = create_texture_map(img_path)
        if texture_map is None:
            continue
        
        # Save texture map
        basename = os.path.basename(img_path)
        name, _ = os.path.splitext(basename)
        out_path = os.path.join(output_dir, f"{name}.png")
        
        cv2.imwrite(out_path, texture_map)
        results.append({
            'texture_path': out_path,
            'score': score
        })
    
    return results

In [ ]:
# Texture map'leri oluştur
from texture_generator_v5 import generate_texture_maps
import pandas as pd

df_labels = pd.read_csv('final_labels.csv')
print(f"İşlenecek görsel: {len(df_labels)}")

results = generate_texture_maps(df_labels, output_dir="texture_maps")

# Training data olarak kaydet
df_training = pd.DataFrame(results)
df_training.to_csv('training_data.csv', index=False)

print(f"\nTexture map oluşturuldu: {len(df_training)}")
print(df_training.head())

## 8. Model Training

In [ ]:
%%writefile train_model_v5.py
"""Training script for v5 pipeline"""

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

# GPU setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def load_and_preprocess(path, score):
    """Load and preprocess image"""
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32)
    img = tf.image.grayscale_to_rgb(img)
    return img, score

def augment(img, score):
    """Data augmentation"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    
    # Random rotation (small)
    angle = tf.random.uniform([], -0.1, 0.1)
    img = tfa_rotate(img, angle) if 'tfa_rotate' in dir() else img
    
    return img, score

def create_dataset(df, batch_size=32, augment_data=False, shuffle=True):
    """Create TensorFlow dataset"""
    paths = df['texture_path'].values
    scores = df['score'].values.astype(np.float32)
    
    dataset = tf.data.Dataset.from_tensor_slices((paths, scores))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df))
    
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment_data:
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

def build_model():
    """Build EfficientNetB0 model"""
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    
    # Unfreeze last 30 layers
    for layer in base.layers[:-30]:
        layer.trainable = False
    
    model = tf.keras.Sequential([
        base,
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.Dense(1, dtype='float32')
    ])
    
    return model

def train(csv_path='training_data.csv', epochs=40, batch_size=32):
    """Train the model"""
    # Load data
    df = pd.read_csv(csv_path)
    print(f"Toplam veri: {len(df)}")
    print(f"Skor: min={df['score'].min():.1f}, max={df['score'].max():.1f}, mean={df['score'].mean():.1f}")
    
    # Split
    df_train, df_val = train_test_split(df, test_size=0.15, random_state=42)
    print(f"Train: {len(df_train)}, Val: {len(df_val)}")
    
    # Datasets
    train_ds = create_dataset(df_train, batch_size=batch_size, augment_data=True, shuffle=True)
    val_ds = create_dataset(df_val, batch_size=batch_size, augment_data=False, shuffle=False)
    
    # Model
    model = build_model()
    
    # Learning rate schedule with warmup
    initial_lr = 1e-4
    warmup_epochs = 3
    
    def lr_schedule(epoch):
        if epoch < warmup_epochs:
            return initial_lr * (epoch + 1) / warmup_epochs
        else:
            # Cosine annealing
            progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
            return initial_lr * 0.5 * (1 + np.cos(np.pi * progress))
    
    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=initial_lr),
        loss=tf.keras.losses.Huber(delta=10.0),
        metrics=['mae']
    )
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.LearningRateScheduler(lr_schedule),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_mae',
            patience=8,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            'best_model.keras',
            monitor='val_mae',
            save_best_only=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_mae',
            factor=0.5,
            patience=4,
            min_lr=1e-6
        )
    ]
    
    # Train
    print(f"\nEğitim başlıyor ({epochs} epoch)...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks
    )
    
    return model, history

if __name__ == '__main__':
    model, history = train()

In [ ]:
# Modeli eğit
from train_model_v5 import train

model, history = train(
    csv_path='training_data.csv',
    epochs=40,
    batch_size=32
)

In [ ]:
# Eğitim grafiği
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# MAE
axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Val')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

print(f"\nEn iyi Val MAE: {min(history.history['val_mae']):.2f}")

## 9. Model Test & Export

In [ ]:
# Validation setinde detaylı test
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Model yükle
model = tf.keras.models.load_model('best_model.keras')

# Veri yükle
df = pd.read_csv('training_data.csv')
_, df_val = train_test_split(df, test_size=0.15, random_state=42)

# Tahmin yap
predictions = []
actuals = []

for _, row in df_val.iterrows():
    img = tf.io.read_file(row['texture_path'])
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32)
    img = tf.image.grayscale_to_rgb(img)
    img = tf.expand_dims(img, 0)
    
    pred = model.predict(img, verbose=0)[0][0]
    predictions.append(np.clip(pred, 0, 100))
    actuals.append(row['score'])

predictions = np.array(predictions)
actuals = np.array(actuals)

# Skor aralıklarına göre analiz
print("=== SKOR ARALIĞINA GÖRE MAE ===")
ranges = [(0, 30), (30, 50), (50, 70), (70, 100)]

for low, high in ranges:
    mask = (actuals >= low) & (actuals < high)
    if mask.sum() > 0:
        mae = np.mean(np.abs(predictions[mask] - actuals[mask]))
        print(f"  {low}-{high}: MAE={mae:.2f} (n={mask.sum()})")

overall_mae = np.mean(np.abs(predictions - actuals))
print(f"\nGENEL MAE: {overall_mae:.2f}")

In [ ]:
# Model export
import shutil

export_dir = 'texture_model_v5'
os.makedirs(export_dir, exist_ok=True)

# Model
model.save(f'{export_dir}/texture_regressor.keras')

# Kalibrasyon modeli
shutil.copy('calibration_model.joblib', f'{export_dir}/')

# Labellar
shutil.copy('final_labels.csv', f'{export_dir}/')
shutil.copy('manual_labels.json', f'{export_dir}/')

# Zip
shutil.make_archive('texture_model_v5', 'zip', export_dir)

print("Export tamamlandı!")
print("İndirilecek dosya: texture_model_v5.zip")

In [ ]:
# Colab'dan indir
from google.colab import files
files.download('texture_model_v5.zip')

## 10. Test with Upload

In [ ]:
# Görsel yükleyerek test et
from google.colab import files
import tensorflow as tf
import cv2
import numpy as np
from IPython.display import display, Image

def create_texture_map_simple(img):
    img = cv2.resize(img, (256, 256))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel = cv2.split(lab)[0]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(l_channel)
    blurred = cv2.GaussianBlur(enhanced, (37, 37), 0)
    texture_map = cv2.add(cv2.subtract(enhanced, blurred), 128)
    return cv2.resize(texture_map, (224, 224))

def predict_score(model, img_path):
    img = cv2.imread(img_path)
    texture_map = create_texture_map_simple(img)
    
    # Model input
    input_img = np.stack([texture_map, texture_map, texture_map], axis=-1)
    input_img = input_img.astype(np.float32)
    input_img = np.expand_dims(input_img, 0)
    
    pred = model.predict(input_img, verbose=0)[0][0]
    return np.clip(pred, 0, 100)

# Model yükle
model = tf.keras.models.load_model('best_model.keras')

print("Görsel yükleyin:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Görseli kaydet
    with open(filename, 'wb') as f:
        f.write(uploaded[filename])
    
    # Tahmin
    score = predict_score(model, filename)
    
    # Sonuç
    print(f"\n{'='*40}")
    display(Image(filename, width=300))
    print(f"TEXTURE SCORE: {score:.1f}/100")
    
    if score >= 70:
        print("Yorum: Pürüzsüz cilt ✓")
    elif score >= 50:
        print("Yorum: Normal doku")
    elif score >= 35:
        print("Yorum: Orta düzey")
    else:
        print("Yorum: Belirgin doku")